In [1]:
from dataclasses import dataclass
from enum import Enum
import random
import time
from typing import Any, Callable, Dict, List, Optional, TypedDict


# ==========================================
# 1. 归一化异常与 ToolResult 契约
# ==========================================
class ErrorKind(Enum):
    RETRYABLE = "retryable"
    NON_RETRYABLE = "non_retryable"
    RATE_LIMITED = "rate_limited"
    TIMEOUT = "timeout"
    OVERLOADED = "overloaded"
    INTERNAL = "internal"


@dataclass
class ToolError:
    kind: ErrorKind
    message: str
    retryable: bool
    status_code: Optional[int] = None
    retry_after: Optional[float] = None


@dataclass
class ToolResult:
    ok: bool
    tool_name: str
    data: Any = None
    error: Optional[ToolError] = None
    attempts: int = 0
    latency_ms: float = 0.0


def normalize_error(
    status_code: Optional[int] = None,
    exc: Optional[Exception] = None,
    retry_after: Optional[float] = None,
    is_bulkhead_rejected: bool = False,
    is_circuit_open: bool = False,
) -> ToolError:
    if is_circuit_open:
        return ToolError(
            kind=ErrorKind.OVERLOADED,
            message="CircuitBreaker is OPEN",
            retryable=False,
        )

    if is_bulkhead_rejected:
        return ToolError(
            kind=ErrorKind.OVERLOADED,
            message="Bulkhead capacity full",
            retryable=False,
            status_code=503,
        )

    if exc is not None:
        if isinstance(exc, TimeoutError):
            return ToolError(
                kind=ErrorKind.TIMEOUT,
                message=str(exc) or "Request execution timed out",
                retryable=True,
            )
        return ToolError(
            kind=ErrorKind.INTERNAL,
            message=f"Internal exception: {type(exc).__name__} - {str(exc)}",
            retryable=False,
        )

    if status_code is not None:
        if status_code == 429:
            return ToolError(
                kind=ErrorKind.RATE_LIMITED,
                message="HTTP 429 Too Many Requests",
                retryable=True,
                status_code=429,
                retry_after=retry_after or 0.1,
            )
        if status_code in (500, 502, 503, 504):
            return ToolError(
                kind=ErrorKind.RETRYABLE,
                message=f"HTTP {status_code} Server Error",
                retryable=True,
                status_code=status_code,
            )
        if status_code in (400, 401, 403, 404, 422):
            return ToolError(
                kind=ErrorKind.NON_RETRYABLE,
                message=f"HTTP {status_code} Client Error",
                retryable=False,
                status_code=status_code,
            )

    return ToolError(
        kind=ErrorKind.INTERNAL,
        message=f"Unknown raw error (status={status_code})",
        retryable=False,
        status_code=status_code,
    )


# ==========================================
# 2. Policy Engine (底层重试与退避策略)
# ==========================================
class PolicyAction(Enum):
    RETRY = "retry"
    FAIL = "fail"
    RATE_LIMIT_WAIT = "rate_limit_wait"
    DEADLINE_EXCEEDED = "deadline_exceeded"


@dataclass
class PolicyDecision:
    action: PolicyAction
    delay: float = 0.0
    reason: str = ""


class PolicyEngine:
    @staticmethod
    def decide(
        error: ToolError,
        retry_count: int,
        max_retries: int,
        base_delay: float,
        remaining_budget: float,
        request_timeout: float,
    ) -> PolicyDecision:
        if not error.retryable:
            return PolicyDecision(action=PolicyAction.FAIL, reason=f"Non-retryable: {error.kind.value}")

        if retry_count >= max_retries:
            return PolicyDecision(action=PolicyAction.FAIL, reason=f"Max retries reached ({max_retries})")

        if error.kind == ErrorKind.RATE_LIMITED:
            computed_delay = error.retry_after if error.retry_after is not None else 0.1
            action = PolicyAction.RATE_LIMIT_WAIT
        else:
            computed_delay = base_delay * (2 ** retry_count) + random.uniform(0.0, 0.01)
            action = PolicyAction.RETRY

        required_budget = computed_delay + request_timeout
        if remaining_budget < required_budget:
            return PolicyDecision(
                action=PolicyAction.DEADLINE_EXCEEDED,
                delay=0.0,
                reason=f"Insufficient remaining budget ({remaining_budget:.3f}s < {required_budget:.3f}s)",
            )

        return PolicyDecision(action=action, delay=computed_delay, reason=f"Allowed {action.value}")


# ==========================================
# 3. CircuitBreaker, Bulkhead & ToolRuntime
# ==========================================
class CircuitBreaker:
    def __init__(self, failure_threshold: int = 3, cooldown: float = 5.0):
        self.state: str = "CLOSED"
        self.failure_count: int = 0
        self.failure_threshold: int = failure_threshold
        self.cooldown: float = cooldown
        self.opened_at: Optional[float] = None

    def can_call(self) -> bool:
        now = time.time()
        if self.state == "OPEN":
            if self.opened_at and (now - self.opened_at >= self.cooldown):
                self.state = "HALF_OPEN"
                return True
            return False
        return True

    def record_success(self):
        self.failure_count = 0
        self.state = "CLOSED"

    def record_failure(self):
        self.failure_count += 1
        if self.state == "HALF_OPEN" or self.failure_count >= self.failure_threshold:
            self.state = "OPEN"
            self.opened_at = time.time()


class Bulkhead:
    def __init__(self, capacity: int):
        self.capacity: int = capacity
        self.in_flight: int = 0

    def try_acquire(self) -> bool:
        if self.in_flight < self.capacity:
            self.in_flight += 1
            return True
        return False

    def release(self) -> None:
        if self.in_flight > 0:
            self.in_flight -= 1


@dataclass
class ToolConfig:
    name: str
    max_retries: int
    base_delay: float
    timeout: float
    breaker: CircuitBreaker
    bulkhead: Bulkhead
    requires_approval: bool = False  # 是否为高危敏感操作（Graph层策略使用）


class ToolRegistry:
    def __init__(self):
        self.configs: Dict[str, ToolConfig] = {}
        self.funcs: Dict[str, Callable] = {}

    def register(self, config: ToolConfig, func: Callable):
        self.configs[config.name] = config
        self.funcs[config.name] = func

    def get_config(self, name: str) -> ToolConfig:
        return self.configs[name]

    def get_func(self, name: str) -> Callable:
        return self.funcs[name]


class ToolRuntime:
    def __init__(self, registry: ToolRegistry):
        self.registry = registry

    def execute(self, tool_name: str, args: Dict[str, Any], deadline: Optional[float] = None) -> ToolResult:
        start_time = time.perf_counter()
        config = self.registry.get_config(tool_name)
        tool_fn = self.registry.get_func(tool_name)

        if deadline is None:
            deadline = time.time() + 10.0

        if not config.breaker.can_call():
            elapsed_ms = (time.perf_counter() - start_time) * 1000
            return ToolResult(
                ok=False,
                tool_name=tool_name,
                data=None,
                error=normalize_error(is_circuit_open=True),
                attempts=0,
                latency_ms=elapsed_ms,
            )

        retry_count = 0

        while True:
            if not config.bulkhead.try_acquire():
                elapsed_ms = (time.perf_counter() - start_time) * 1000
                return ToolResult(
                    ok=False,
                    tool_name=tool_name,
                    data=None,
                    error=normalize_error(is_bulkhead_rejected=True),
                    attempts=retry_count,
                    latency_ms=elapsed_ms,
                )

            raw_result = None
            caught_exc = None
            try:
                raw_result = tool_fn(args)
            except Exception as e:
                caught_exc = e
            finally:
                config.bulkhead.release()

            status_code = raw_result.get("status") if isinstance(raw_result, dict) else None
            retry_after = raw_result.get("retry_after") if isinstance(raw_result, dict) else None

            if caught_exc is not None or (status_code and status_code != 200):
                tool_error = normalize_error(status_code=status_code, exc=caught_exc, retry_after=retry_after)
            else:
                tool_error = None

            if tool_error is None:
                config.breaker.record_success()
                elapsed_ms = (time.perf_counter() - start_time) * 1000
                return ToolResult(
                    ok=True,
                    tool_name=tool_name,
                    data=raw_result,
                    error=None,
                    attempts=retry_count + 1,
                    latency_ms=elapsed_ms,
                )

            config.breaker.record_failure()

            remaining_budget = deadline - time.time()
            decision = PolicyEngine.decide(
                error=tool_error,
                retry_count=retry_count,
                max_retries=config.max_retries,
                base_delay=config.base_delay,
                remaining_budget=remaining_budget,
                request_timeout=config.timeout,
            )

            if decision.action in (PolicyAction.FAIL, PolicyAction.DEADLINE_EXCEEDED):
                elapsed_ms = (time.perf_counter() - start_time) * 1000
                return ToolResult(
                    ok=False,
                    tool_name=tool_name,
                    data=raw_result,
                    error=tool_error if decision.action == PolicyAction.FAIL else ToolError(
                        kind=ErrorKind.TIMEOUT,
                        message=f"Deadline Exceeded: {decision.reason}",
                        retryable=False,
                    ),
                    attempts=retry_count + 1,
                    latency_ms=elapsed_ms,
                )

            if decision.action in (PolicyAction.RETRY, PolicyAction.RATE_LIMIT_WAIT):
                time.sleep(decision.delay)
                retry_count += 1


# ==========================================
# 4. Agent State & Mock LLM Decision Layer
# ==========================================
class AgentState(TypedDict):
    user_query: str
    messages: List[Dict[str, Any]]
    pending_tool_name: Optional[str]
    pending_tool_args: Optional[Dict[str, Any]]
    latest_tool_result: Optional[ToolResult]
    final_answer: Optional[str]
    is_finished: bool
    deadline: float


def mock_llm_agent(state: AgentState) -> Dict[str, Any]:
    """
    Mock LLM 决策层：
    根据当前已收集的工具调用结果 (messages)，动态决定下一步是调用新工具还是给出 Final Answer。
    """
    history_tools = [m["tool_name"] for m in state["messages"] if m.get("type") == "tool_result" and m["ok"]]

    # 场景逻辑推演：
    # 1. 还没有查过请假政策 -> 调用 search_policy
    if "search_policy" not in history_tools:
        return {
            "pending_tool_name": "search_policy",
            "pending_tool_args": {"policy_type": "annual_leave"},
            "is_finished": False,
        }

    # 2. 查过政策但还没查员工信息 -> 调用 get_employee
    if "get_employee" not in history_tools:
        return {
            "pending_tool_name": "get_employee",
            "pending_tool_args": {"employee_id": "EMP_1001"},
            "is_finished": False,
        }

    # 3. 政策和员工信息都准备就绪 -> 提交请假 submit_leave
    if "submit_leave" not in history_tools:
        return {
            "pending_tool_name": "submit_leave",
            "pending_tool_args": {"employee_id": "EMP_1001", "days": 3},
            "is_finished": False,
        }

    # 4. 所有工具准备完成 -> 生成最终回答
    return {
        "pending_tool_name": None,
        "pending_tool_args": None,
        "final_answer": "您的 3 天年假申请已成功提交并自动审核通过！",
        "is_finished": True,
    }


# ==========================================
# 5. Graph 策略检查层与工具执行节点
# ==========================================
def graph_policy_check(tool_name: str, registry: ToolRegistry) -> bool:
    """
    Graph 层的业务策略检查（区别于 Runtime 内部的防御策略）：
    如：高危敏感操作阻断/拦截审核。
    """
    config = registry.get_config(tool_name)
    if config.requires_approval:
        print(f"  [Graph Policy Check] Warning: '{tool_name}' is a sensitive action (requires approval).")
    return True


def execute_tool_node(state: AgentState, runtime: ToolRuntime, registry: ToolRegistry) -> Dict[str, Any]:
    tool_name = state["pending_tool_name"]
    tool_args = state["pending_tool_args"]

    # 1. Graph/Policy 检查
    passed = graph_policy_check(tool_name, registry)
    if not passed:
        err_res = ToolResult(
            ok=False,
            tool_name=tool_name,
            error=ToolError(kind=ErrorKind.NON_RETRYABLE, message="Graph Policy Rejected", retryable=False),
        )
        return {"latest_tool_result": err_res}

    # 2. ToolRuntime 执行
    result = runtime.execute(tool_name=tool_name, args=tool_args, deadline=state["deadline"])

    # 3. 记录日志到消息轨迹 (messages)
    new_message = {
        "type": "tool_result",
        "tool_name": tool_name,
        "ok": result.ok,
        "data": result.data,
        "attempts": result.attempts,
    }

    return {
        "latest_tool_result": result,
        "messages": state["messages"] + [new_message],
        "pending_tool_name": None,
        "pending_tool_args": None,
    }


# ==========================================
# 6. Minimal Multi-Tool Agent Loop 运行引擎
# ==========================================
def run_agent_loop(query: str, runtime: ToolRuntime, registry: ToolRegistry) -> AgentState:
    state: AgentState = {
        "user_query": query,
        "messages": [],
        "pending_tool_name": None,
        "pending_tool_args": None,
        "latest_tool_result": None,
        "final_answer": None,
        "is_finished": False,
        "deadline": time.time() + 10.0,
    }

    step = 1
    print(f"\n================ Start Multi-Tool Agent Loop ================")
    print(f"User Query: '{query}'\n")

    while not state["is_finished"]:
        print(f"--- Step {step}: Agent Thinking ---")
        
        # 1. LLM / Decision Layer 决策
        decision = mock_llm_agent(state)
        state.update(decision)

        if state["is_finished"]:
            print(f"Agent Action: Final Answer Generated.")
            break

        tool_name = state["pending_tool_name"]
        tool_args = state["pending_tool_args"]
        print(f"Agent Decision -> Selected Tool: [{tool_name}] with args: {tool_args}")

        # 2. Tool Execution Node (包含 Policy Check + Runtime Execute)
        node_update = execute_tool_node(state, runtime, registry)
        state.update(node_update)

        latest_res = state["latest_tool_result"]
        print(f"Runtime Executed -> Tool: [{latest_res.tool_name}] | OK: {latest_res.ok} | Attempts: {latest_res.attempts} | Data: {latest_res.data}")
        
        if not latest_res.ok:
            print(f"  [Notice] Tool Execution Failed with Error: {latest_res.error.message}")
            if latest_res.error.kind == ErrorKind.NON_RETRYABLE:
                print("  [Terminal Failure] Agent terminating loop due to non-retryable error.")
                state["final_answer"] = f"办理失败，原因：{latest_res.error.message}"
                state["is_finished"] = True
                break

        step += 1

    print(f"\n================ Agent Loop Finished ================")
    print(f"Final Output: {state['final_answer']}")
    return state


# ==========================================
# 7. Mock 工具实现与测试验证
# ==========================================
def mock_search_policy(args: dict) -> dict:
    return {"status": 200, "policy": "员工满1年享有5天带薪年假，需提前申请。"}


class MockGetEmployee:
    def __init__(self, sequence: List[Any]):
        self.sequence = sequence

    def __call__(self, args: dict) -> dict:
        item = self.sequence.pop(0) if self.sequence else 200
        if isinstance(item, int):
            if item == 200:
                return {"status": 200, "employee": {"id": args["employee_id"], "name": "张三", "leave_balance": 5}}
            return {"status": item, "message": "Server Error"}
        raise item


def mock_submit_leave(args: dict) -> dict:
    return {"status": 200, "approval_id": "APP_99821", "result": "APPROVED"}


if __name__ == "__main__":
    # 注册组件与工具配置
    registry = ToolRegistry()
    runtime = ToolRuntime(registry)

    # 1. search_policy
    registry.register(
        ToolConfig(name="search_policy", max_retries=1, base_delay=0.01, timeout=1.0, breaker=CircuitBreaker(), bulkhead=Bulkhead(5)),
        mock_search_policy,
    )

    # 2. get_employee (模拟第一次 503 触发 Runtime 重试，第二次 200 成功)
    registry.register(
        ToolConfig(name="get_employee", max_retries=2, base_delay=0.01, timeout=1.0, breaker=CircuitBreaker(), bulkhead=Bulkhead(5)),
        MockGetEmployee([503, 200]),
    )

    # 3. submit_leave (高危操作配置)
    registry.register(
        ToolConfig(name="submit_leave", max_retries=1, base_delay=0.01, timeout=1.0, breaker=CircuitBreaker(), bulkhead=Bulkhead(5), requires_approval=True),
        mock_submit_leave,
    )

    # 运行 Agent 循环测试
    final_state = run_agent_loop("我要请3天年假，帮我查询政策并提交申请", runtime, registry)


================ Start Multi-Tool Agent Loop ================
User Query: '我要请3天年假，帮我查询政策并提交申请'

--- Step 1: Agent Thinking ---
Agent Decision -> Selected Tool: [search_policy] with args: {'policy_type': 'annual_leave'}
Runtime Executed -> Tool: [search_policy] | OK: True | Attempts: 1 | Data: {'status': 200, 'policy': '员工满1年享有5天带薪年假，需提前申请。'}
--- Step 2: Agent Thinking ---
Agent Decision -> Selected Tool: [get_employee] with args: {'employee_id': 'EMP_1001'}
Runtime Executed -> Tool: [get_employee] | OK: True | Attempts: 2 | Data: {'status': 200, 'employee': {'id': 'EMP_1001', 'name': '张三', 'leave_balance': 5}}
--- Step 3: Agent Thinking ---
Agent Decision -> Selected Tool: [submit_leave] with args: {'employee_id': 'EMP_1001', 'days': 3}
  [Graph Policy Check] Warning: 'submit_leave' is a sensitive action (requires approval).
Runtime Executed -> Tool: [submit_leave] | OK: True | Attempts: 1 | Data: {'status': 200, 'approval_id': 'APP_99821', 'result': 'APPROVED'}
--- Step 4: Agent Thi